<a href="https://colab.research.google.com/github/kkinca/medical-rag-assistant/blob/main/RAG_Project_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# ==============================
# 1. INSTALL REQUIRED PACKAGES
# ==============================

!pip install -q --upgrade pip

!pip install -q \
    langchain==0.3.7 \
    langchain-community==0.3.7 \
    langchain-core==0.3.17 \
    langchain-text-splitters==0.3.2 \
    chromadb==0.5.5 \
    sentence-transformers==3.0.1 \
    pypdf \
    llama-cpp-python==0.2.90

In [2]:
print("Install complete.")
print("IMPORTANT: Now go to Runtime > Restart session, then continue running the notebook from the next cell.")

Install complete.
IMPORTANT: Now go to Runtime > Restart session, then continue running the notebook from the next cell.


In [3]:
# ==============================
# 2. IMPORT LIBRARIES
# ==============================

import os
import json
import warnings

warnings.filterwarnings("ignore")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.llms import LlamaCpp
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.embeddings.base import Embeddings

from sentence_transformers import SentenceTransformer

In [4]:
# ==============================
# 3. FILE PATHS
# ==============================

PDF_PATH = "/content/medical_diagnosis_manual.pdf"
CHROMA_DIR = "medical_db"

print("PDF exists:", os.path.exists(PDF_PATH))

PDF exists: True


In [5]:
# ==============================
# 4. LOAD PDF
# ==============================

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print(f"Loaded {len(documents)} pages from the medical manual.")

Loaded 4114 pages from the medical manual.


In [6]:
# ==============================
# 5. SPLIT TEXT INTO CHUNKS
# ==============================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

print(f"Created {len(docs)} chunks.")

Created 18207 chunks.


In [7]:
# ==============================
# 6. EMBEDDING MODEL
# ==============================

class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        return self.model.encode(text).tolist()

embedding_model = SentenceTransformerEmbeddings("all-MiniLM-L6-v2")

print("Embedding model loaded.")

Embedding model loaded.


In [8]:
# ==============================
# 7. CREATE VECTOR DATABASE
# ==============================

vectordb = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory=CHROMA_DIR
)

retriever = vectordb.as_retriever(search_kwargs={"k": 3})

print("Vector database created and retriever ready.")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector database created and retriever ready.


In [9]:
# ==============================
# 8A. DOWNLOAD THE GGUF MODEL
# ==============================

from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="TheBloke/Mistral-7B-Instruct-v0.2-GGUF",
    filename="mistral-7b-instruct-v0.2.Q6_K.gguf"
)

print("Model downloaded to:", model_path)

Model downloaded to: /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q6_K.gguf


In [10]:
# ==============================
# 8. LOAD LOCAL LLM
# ==============================

llm = LlamaCpp(
    model_path=model_path,
    temperature=0.2,
    max_tokens=350,
    top_p=0.9,
    top_k=40,
    n_ctx=4096,
    verbose=False
)

print("LLM loaded.")

LLM loaded.


In [11]:
# ==============================
# 9. LLM-ONLY BASELINE
# ==============================

baseline_prompt = PromptTemplate(
    input_variables=["question"],
    template="""
You are a clinical medical assistant.
Provide structured, step-by-step, medically accurate responses.
Avoid speculation.
Clearly distinguish symptoms, diagnosis, and treatment.
If uncertain, state limitations explicitly.

Question: {question}

Answer:
"""
)

def run_llm_baseline(question):
    prompt = baseline_prompt.format(question=question)
    return llm.invoke(prompt)

In [12]:
question = "What is the recommended management for sepsis?"
baseline_answer = run_llm_baseline(question)
print(baseline_answer)

1. Recognize early signs of sepsis, including fever, chills, rapid heart rate, rapid breathing, confusion, or decreased urine output.
2. Obtain a complete medical history and perform a physical examination to identify any potential sources of infection.
3. Initiate appropriate antibiotic therapy based on the suspected source of infection and local antibiotic resistance patterns.
4. Provide adequate fluid resuscitation to maintain adequate tissue perfusion and organ function. This may involve administering intravenous crystalloid solutions or colloid solutions, depending on the patient's clinical condition and response to fluid resuscitation.
5. Monitor the patient closely for signs of improvement or deterioration, and adjust treatment plans accordingly.
6. Provide supportive care measures as needed, such as oxygen therapy, vasopressor medications, or mechanical ventilation, to help maintain adequate tissue perfusion and organ function in patients with sepsis.


In [13]:
# ==============================
# 10. RAG CHAIN
# ==============================

rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a clinical medical assistant.
Answer the question using only the provided context from the Merck Manual.
If the information is not available in the context, clearly state that it is not found.

Context:
{context}

Question:
{question}

Provide a structured, medically accurate answer based strictly on the context.
"""
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": rag_prompt},
    return_source_documents=True
)

In [14]:
question = "What is the recommended management for sepsis?"
rag_result = qa_chain.invoke({"query": question})

print("RAG ANSWER:\n")
print(rag_result["result"])

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


RAG ANSWER:


Answer:
The management for sepsis involves prompt recognition and immediate initiation of appropriate therapy. The following steps are recommended based on the context provided:

1. Obtain specimens of blood, body fluids, and wound sites for Gram stain and culture before administering antibiotics.

2. Initiate empiric antibiotic therapy as soon as possible after suspecting sepsis. The choice of antibiotics should be based on the suspected source, clinical setting, knowledge or suspicion of causative organisms and of sensitivity patterns common to that specific inpatient unit, and previous culture results.

3. Provide supportive care, including adequate fluid resuscitation, maintenance of adequate oxygenation and ventilation, and correction of electrolyte imbalances as needed.

4. Monitor the patient closely for signs of clinical improvement or deterioration. Adjust the treatment plan as necessary based on the patient's response to therapy and any new developments or compl

In [15]:
print("\nRETRIEVED SOURCES:\n")

for i, doc in enumerate(rag_result["source_documents"], 1):
    print(f"--- Source {i} ---")
    print(doc.page_content[:1000])
    print("\n")


RETRIEVED SOURCES:

--- Source 1 ---
Parenteral antibiotics should be given after specimens of blood, body fluids, and wound sites have been
taken for Gram stain and culture. Very prompt empiric therapy, started immediately after suspecting
sepsis, is essential and may be lifesaving. Antibiotic selection requires an educated guess based on the
suspected source, clinical setting, knowledge or suspicion of causative organisms and of sensitivity
patterns common to that specific inpatient unit, and previous culture results.
One regimen for septic shock of unknown cause is gentamicin or tobramycin 5.1 mg/kg IV once/day plus
a 3rd-generation cephalosporin (cefotaxime 2 g q 6 to 8 h or ceftriaxone 2 g once/day or, if 
Pseudomonas
is suspected, ceftazidime 2 g IV q 8 h). Alternatively, ceftazidime plus a fluoroquinolone (eg, ciprofloxacin)
may be used. Monotherapy with maximal therapeutic doses of ceftazidime (2 g IV q 8 h) or imipenem (1 g
IV q 6 h) may be effective but is not recommended.



In [16]:
# ==============================
# 11. SAMPLE QUESTIONS
# ==============================

questions = [
    "What is the recommended management for sepsis?",
    "What are the symptoms and treatment options for appendicitis?",
    "What causes sudden patchy hair loss and how is it treated?",
    "How is traumatic brain injury managed?",
    "What precautions and recovery steps are recommended for a leg fracture?"
]

for q in questions:
    result = qa_chain.invoke({"query": q})
    print("=" * 80)
    print("QUESTION:", q)
    print("\nANSWER:\n", result["result"])
    print("\n")

QUESTION: What is the recommended management for sepsis?

ANSWER:
 
Answer:
The recommended management for sepsis includes prompt recognition and immediate initiation of appropriate antimicrobial therapy after obtaining specimens for Gram stain and culture. The choice of antibiotics depends on the suspected source, clinical setting, knowledge or suspicion of causative organisms and their sensitivity patterns, and previous culture results. In addition to antimicrobial therapy, supportive measures such as adequate fluid resuscitation, oxygen therapy, and nutritional support may be necessary. Close monitoring of vital signs, urine output, and mental status is essential for early detection and prompt management of any complications or deterioration in clinical condition.


QUESTION: What are the symptoms and treatment options for appendicitis?

ANSWER:
 
Answer:
Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch located in the lower right side 

In [17]:
# ==============================
# 12. EVALUATION PROMPTS
# ==============================

groundedness_prompt = """
Evaluate whether the answer is fully supported by the provided context.

Score from 1 to 5:
1 = Not supported by context
3 = Partially supported
5 = Fully grounded in context

Provide structured reasoning for the assigned score.

Question: {question}

Context:
{context}

Answer:
{answer}
"""

relevance_prompt = """
Evaluate how directly and completely the answer addresses the user's question.

Score from 1 to 5:
1 = Not relevant
3 = Partially relevant
5 = Fully relevant and complete

Provide structured reasoning for the assigned score.

Question: {question}

Answer:
{answer}
"""

## Question Answering using LLM

### Response

In [18]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [19]:
question = "What treatment options are available for managing hypertension?"
baseline_answer = run_llm_baseline(question)
print(baseline_answer)

1. Medications: The most common treatment for hypertension is medication. There are several classes of drugs used to treat hypertension, including diuretics, beta blockers, ACE inhibitors, and calcium channel blockers.
2. Lifestyle modifications: In addition to medications, lifestyle modifications can also help manage hypertension. These modifications include eating a healthy diet rich in fruits, vegetables, whole grains, and lean protein sources; limiting sodium intake to less than 2,300 milligrams per day or even less if you have hypertension or diabetes; maintaining a healthy weight; getting regular physical activity; reducing stress through relaxation techniques such as deep breathing, meditation, yoga, or progressive muscle relaxation; and avoiding alcohol or limiting alcohol consumption to no more than one drink per day for women and no more than two drinks per day for men.
3. Complementary therapies: Some complementary therapies may also help manage hypertension. These therapies

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [20]:
question = "What is the protocol for managing sepsis in a critical care unit?"
baseline_answer = run_llm_baseline(question)
print(baseline_answer)


1. Recognition of Sepsis:
   - Look for signs of infection (wound infection, urinary tract infection, etc.)
   - Check for systemic inflammatory response syndrome (SIRS) criteria:
      - Body temperature >38°C (100.4°F) or <36°C (96.8°F))
      - Heart rate >90 beats per minute
      - Respiratory rate >20 breaths per minute or arterial carbon dioxide tension (PaCO2) >45 mmHg

2. Initial Assessment and Treatment:
   - Initiate high-flow oxygen therapy via a non-rebreather mask or an advanced airway (endotracheal tube or tracheostomy tube).
   - Administer intravenous (IV) fluids to maintain adequate tissue perfusion. The initial fluid resuscitation should be 30 mL/kg of body weight in the first hour, followed by reassessment and further fluid administration as needed.
   - Initiate broad-spectrum antibiotics based on the suspected source of infection.
   - Monitor vital signs closely and initiate appropriate interventions to maintain adequate tissue perfusion and oxygenation.
   - Pr

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [21]:
# ==============================
# BASELINE TEST QUESTION 2
# ==============================

question = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
baseline_answer = run_llm_baseline(question)

print("QUESTION:\n", question)
print("\nBASELINE ANSWER:\n")
print(baseline_answer)

QUESTION:
 What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

BASELINE ANSWER:

Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch located in the lower right abdomen.

Common symptoms of appendicitis include:
1. Sudden onset of pain, usually starting around the navel and then shifting to the lower right abdomen.
2. Loss of appetite and/or feeling sick to your stomach (nausea).
3. Fever, often with a temperature above 101 degrees Fahrenheit (38.3 degrees Celsius).
4. Abdominal swelling and/or tenderness, especially in the lower right quadrant of the abdomen.
5. Passing gas (flatulence) or having diarrhea may also occur, but these symptoms are less common than the others listed above.

It is important to note that not all cases of appendicitis present with all of the symptoms listed above. In some cases, the symptoms may be mild or atypical, making i

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [22]:
# ==============================
# BASELINE TEST QUESTION 3
# ==============================

question = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
baseline_answer = run_llm_baseline(question)

print("QUESTION:\n", question)
print("\nBASELINE ANSWER:\n")
print(baseline_answer)

QUESTION:
 What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

BASELINE ANSWER:


Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. The exact cause of alopecia areata is not fully understood, but it is believed to be related to genetics and environmental factors.

As for the effective treatments or solutions for addressing sudden patchy hair loss, there is no one-size-fits-all answer, as the most appropriate treatment approach depends on various individual factors, such as the extent and duration of hair loss, the underlying cause of the hair loss, and the overall health and medical history of the person.

That being said, some common treatments for alopecia areata include:

1. Corticosteroids: These are anti-inflammatory medications that can help reduce inflammation in the affecte

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [23]:
# ==============================
# BASELINE TEST QUESTION 4
# ==============================

question = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
baseline_answer = run_llm_baseline(question)

print("QUESTION:\n", question)
print("\nBASELINE ANSWER:\n")
print(baseline_answer)

QUESTION:
 What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

BASELINE ANSWER:

1. Initial Treatment: The primary focus is on stabilizing the patient's condition and preventing further damage to the brain. This may include administering oxygen, controlling seizures with medications, managing blood pressure, and providing adequate hydration and nutrition.
2. Rehabilitation: Depending on the extent of the injury and resulting impairment, rehabilitation may be necessary to help the patient regain lost function and improve overall quality of life. This may include physical therapy to address motor deficits, occupational therapy to help with activities of daily living (ADLs), speech-language therapy to address communication difficulties, and cognitive rehabilitation to help with memory, attention, and executive functioning deficits.
3. Medications: Depending on the specific symp

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [24]:
# ==============================
# BASELINE TEST QUESTION 5
# ==============================

question = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
baseline_answer = run_llm_baseline(question)

print("QUESTION:\n", question)
print("\nBASELINE ANSWER:\n")
print(baseline_answer)

QUESTION:
 What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

BASELINE ANSWER:


Precautions:
1. Immobilize the affected leg using a splint or cast to prevent further damage and promote healing.
2. Apply ice packs intermittently (every 1-2 hours) for the first 24-48 hours to reduce swelling, pain, and inflammation.
3. Elevate the affected leg above heart level whenever possible to minimize swelling and promote proper blood flow.
4. Avoid putting weight on the affected leg or bearing any pressure on it until it has healed completely.
5. Use crutches, a walker, or other assistive devices as needed to help you get around safely and comfortably without putting undue stress on your injured leg.
6. Keep the affected leg clean and dry at all times to prevent infection and promote proper healing.
7. Follow a healthy, balanced diet that is rich in essential nutrients, v

## Question Answering using LLM with Prompt Engineering

In [25]:
system_prompt = "You are a knowledgeable medical assistant. Provide clear, structured, and professional answers based on medical knowledge. Keep responses informative and concise."

# **Prompt Engineering – Parameter Tuning Experiments**

## Prompt Engineering – Parameter Tuning

Different parameter settings were tested to improve structure, completeness, and clinical consistency in LLM-only responses.

The final configuration selected was:

- Temperature = 0.2
- Max Tokens = 350
- Top-p = 0.9
- Top-k = 40

This setting provided the best balance of step-by-step clarity, response completeness, and reduced variability.

In [30]:
# ==============================
# PROMPT ENGINEERING RESULTS
# ==============================

prompt_tuning_results = [
    {"Run": 1, "Temperature": 0.0, "MaxTokens": 250, "Top-p": 0.90, "Top-k": 40, "Outcome": "Balanced length; Stepwise; More assertive"},
    {"Run": 2, "Temperature": 0.3, "MaxTokens": 350, "Top-p": 0.95, "Top-k": 50, "Outcome": "Balanced length; Stepwise; Cautious"},
    {"Run": 3, "Temperature": 0.2, "MaxTokens": 350, "Top-p": 0.90, "Top-k": 40, "Outcome": "Balanced length; Stepwise; Cautious"},
    {"Run": 4, "Temperature": 0.6, "MaxTokens": 300, "Top-p": 0.95, "Top-k": 50, "Outcome": "Balanced length; Stepwise; More assertive"},
    {"Run": 5, "Temperature": 0.2, "MaxTokens": 200, "Top-p": 0.85, "Top-k": 30, "Outcome": "Balanced length; Stepwise; More assertive"},
]

for row in prompt_tuning_results:
    print(row)

{'Run': 1, 'Temperature': 0.0, 'MaxTokens': 250, 'Top-p': 0.9, 'Top-k': 40, 'Outcome': 'Balanced length; Stepwise; More assertive'}
{'Run': 2, 'Temperature': 0.3, 'MaxTokens': 350, 'Top-p': 0.95, 'Top-k': 50, 'Outcome': 'Balanced length; Stepwise; Cautious'}
{'Run': 3, 'Temperature': 0.2, 'MaxTokens': 350, 'Top-p': 0.9, 'Top-k': 40, 'Outcome': 'Balanced length; Stepwise; Cautious'}
{'Run': 4, 'Temperature': 0.6, 'MaxTokens': 300, 'Top-p': 0.95, 'Top-k': 50, 'Outcome': 'Balanced length; Stepwise; More assertive'}
{'Run': 5, 'Temperature': 0.2, 'MaxTokens': 200, 'Top-p': 0.85, 'Top-k': 30, 'Outcome': 'Balanced length; Stepwise; More assertive'}


## Data Preparation for RAG

In [ ]:
## Data Preparation for RAG

The Merck Manual PDF was loaded, split into 1000-character chunks with 200-character overlap, embedded using `all-MiniLM-L6-v2`, and stored in a Chroma vector database for similarity-based retrieval.

### Loading the Data

In [31]:
# ==============================
# LOAD PDF
# ==============================

PDF_PATH = "/content/medical_diagnosis_manual.pdf"

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print(f"Loaded {len(documents)} pages from the medical manual.")

Loaded 4114 pages from the medical manual.


### Data Chunking

In [32]:
# ==============================
# SPLIT TEXT INTO CHUNKS
# ==============================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

print(f"Created {len(docs)} chunks.")

Created 18207 chunks.


In [33]:
# Preview one chunk
print(docs[0].page_content[:1000])

karenruthwood@gmail.com
IAD3SMLQRC
This file is meant for personal use by karenruthwood@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.


### Embedding

In [34]:
# ==============================
# EMBEDDING MODEL
# ==============================

class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        return self.model.encode(text).tolist()

embedding_model = SentenceTransformerEmbeddings("all-MiniLM-L6-v2")

print("Embedding model loaded.")

Embedding model loaded.


### Vector Database

In [35]:
# ==============================
# CREATE VECTOR DATABASE
# ==============================

vectordb = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory="medical_db"
)

retriever = vectordb.as_retriever(search_kwargs={"k": 3})

print("Vector database created and retriever ready.")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector database created and retriever ready.


### Retriever

In [36]:
# Quick retriever test
test_docs = retriever.get_relevant_documents("What is sepsis?")
print(f"Retrieved {len(test_docs)} documents.")
print(test_docs[0].page_content[:1000])

Retrieved 3 documents.
A spectrum of severity exists (see
Table 227-1
).
Sepsis
 is infection accompanied by an acute inflammatory reaction with systemic manifestations
associated with release into the bloodstream of numerous endogenous mediators of inflammation. Acute
pancreatitis and major trauma, including burns, may manifest with signs of sepsis. The inflammatory
reaction typically manifests with ≥ 2 of the following:
• Temperature > 38°C or < 36°C
• 
Heart rate > 90 beats/min
• Respiratory rate > 20 breaths/min or PaCO
2
 < 32 mm Hg
• WBC count > 12,000 cells/
μ
L or < 4,000 cells/
μ
L or > 10% immature forms
However, these criteria are now viewed as suggestive but not sufficiently precise to be diagnostic.
Severe sepsis
 is sepsis accompanied by signs of failure of at least one organ. Cardiovascular failure is
typically manifested by hypotension, respiratory failure by hypoxemia, renal failure by oliguria, and
hematologic failure by coagulopathy.
Septic shock


/tmp/ipykernel_25982/509136318.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  test_docs = retriever.get_relevant_documents("What is sepsis?")


### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [37]:
# ==============================
# RAG PROMPT TEMPLATE
# ==============================

rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a clinical medical assistant.
Answer the question using only the provided context from the Merck Manual.
If the information is not available in the context, clearly state that it is not found.

Context:
{context}

Question:
{question}

Provide a structured, medically accurate answer based strictly on the context.
"""
)

### Response Function

In [38]:
# ==============================
# RAG CHAIN
# ==============================

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": rag_prompt},
    return_source_documents=True
)

print("RAG chain ready.")

RAG chain ready.


## Question Answering using RAG

In [39]:
# ==============================
# RAG SAMPLE QUESTIONS
# ==============================

questions = [
    "What is the recommended management for sepsis?",
    "What are the symptoms and treatment options for appendicitis?",
    "What causes sudden patchy hair loss and how is it treated?",
    "How is traumatic brain injury managed?",
    "What precautions and recovery steps are recommended for a leg fracture?"
]

for q in questions:
    result = qa_chain.invoke({"query": q})
    print("=" * 80)
    print("QUESTION:", q)
    print("\nANSWER:\n", result["result"])
    print("\n")

QUESTION: What is the recommended management for sepsis?

ANSWER:
 
Answer:
The management of sepsis involves prompt recognition and immediate initiation of appropriate antimicrobial therapy. This should be done after specimens of blood, body fluids, and wound sites have been taken for Gram stain and culture. The antibiotic selection requires an educated guess based on the suspected source, clinical setting, knowledge or suspicion of causative organisms and of sensitivity patterns common to that specific inpatient unit, and previous culture results. One regimen for septic shock of unknown cause is gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin (cefotaxime 2 g q 6 to 8 h or ceftriaxone 2 g once/day or, if Pseudomonas is suspected, ceftazidime 2 g IV q 8 h). Alternatively, ceftazidime plus a fluoroquinolone (eg, ciprofloxacin) may be used. Monotherapy with maximal therapeutic doses of ceftazidime (2 g IV q 8 h) or imipenem (1 g IV q 6 h) may be effecti

## RAG Parameter Tuning

## RAG Parameter Tuning

Different values for `k`, `temperature`, and `max_tokens` were tested to balance completeness, focus, and groundedness. The final configuration selected was:

- `k = 3`
- `temperature = 0.2`
- `max_tokens = 350`

This setting provided the best tradeoff between contextual completeness and reduced redundancy.

In [42]:
tuning_results = [
    {"Run": 1, "k": 1, "temperature": 0.2, "max_tokens": 300, "Observation": "Insufficient contextual grounding"},
    {"Run": 2, "k": 3, "temperature": 0.2, "max_tokens": 350, "Observation": "Best balance of completeness and focus"},
    {"Run": 3, "k": 5, "temperature": 0.2, "max_tokens": 350, "Observation": "Too much overlap and verbosity"},
    {"Run": 4, "k": 3, "temperature": 0.4, "max_tokens": 350, "Observation": "Slight variability in phrasing"},
    {"Run": 5, "k": 3, "temperature": 0.2, "max_tokens": 250, "Observation": "Response too short and incomplete"},
]

for row in tuning_results:
    print(row)

{'Run': 1, 'k': 1, 'temperature': 0.2, 'max_tokens': 300, 'Observation': 'Insufficient contextual grounding'}
{'Run': 2, 'k': 3, 'temperature': 0.2, 'max_tokens': 350, 'Observation': 'Best balance of completeness and focus'}
{'Run': 3, 'k': 5, 'temperature': 0.2, 'max_tokens': 350, 'Observation': 'Too much overlap and verbosity'}
{'Run': 4, 'k': 3, 'temperature': 0.4, 'max_tokens': 350, 'Observation': 'Slight variability in phrasing'}
{'Run': 5, 'k': 3, 'temperature': 0.2, 'max_tokens': 250, 'Observation': 'Response too short and incomplete'}


## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [43]:
# ==============================
# GROUNDEDNESS EVALUATION PROMPT
# ==============================

groundedness_rater_system_message = """
You are a strict medical QA evaluator for a Retrieval-Augmented Generation (RAG) system.
Your job is to judge GROUNDEDNESS only.

Definition:
- Groundedness = how well the answer is supported by the provided context.
- If the answer contains claims that are not present in the context, groundedness drops.
- If the answer contradicts the context, groundedness is very low.

Return ONLY a JSON object with exactly these keys:
{
  "score": integer 1-5,
  "verdict": "<one of: grounded | partially_grounded | not_grounded>",
  "rationale": "<1-3 sentences, cite phrases from the context when possible>"
}

Scoring:
5 = Fully supported by context, no extra claims.
4 = Mostly supported, minor extrapolation.
3 = Some support, but multiple unsupported additions.
2 = Little support, mostly hallucinated.
1 = Not supported or contradicts context.
"""

In [44]:
# ==============================
# RELEVANCE EVALUATION PROMPT
# ==============================

relevance_rater_system_message = """
You are a strict medical QA evaluator for a Retrieval-Augmented Generation (RAG) system.
Your job is to judge RELEVANCE only.

Definition:
- Relevance = whether the answer directly addresses the question asked.
- Do NOT judge truthfulness unless it affects relevance.

Return ONLY a JSON object with exactly these keys:
{
  "score": integer 1-5,
  "verdict": "<one of: relevant | partially_relevant | not_relevant>",
  "rationale": "<1-3 sentences>"
}

Scoring:
5 = Fully relevant and complete.
4 = Relevant with minor omissions.
3 = Partially relevant.
2 = Weakly relevant.
1 = Not relevant.
"""

## RAG Parameter Tuning

Different values for `k`, `temperature`, and `max_tokens` were tested to balance completeness, focus, and groundedness. The final configuration selected was:

- `k = 3`
- `temperature = 0.2`
- `max_tokens = 350`

This setting provided the best tradeoff between contextual completeness and reduced redundancy.